# Tutorial 3: Document Processing with LangChain

In this tutorial, we'll explore document processing techniques using LangChain. We'll cover loading and parsing documents, text splitting, building a simple question-answering system, and implementing semantic search.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

load_dotenv()

llm = ChatGroq(model_name='llama-3.1-8b-instant', temperature=0.7)

# Use OllamaEmbeddings if available, otherwise FakeEmbeddings
try:
    from langchain_ollama import OllamaEmbeddings
    embedding_model = OllamaEmbeddings(model='all-minilm', base_url=os.getenv('OLLAMA_EMBEDDING_URL'))
    _ = embedding_model.embed_query('test')
except Exception:
    from langchain_community.embeddings import FakeEmbeddings
    embedding_model = FakeEmbeddings(size=384)
    print('Ollama not available — using FakeEmbeddings for demo')

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Ollama not available — using FakeEmbeddings for demo
Setup complete.


## 1. Loading and Parsing Documents

In [2]:
# Load a single document
loader = TextLoader("sample_documents/sample1.txt")
document = loader.load()

print(f"Content of sample1.txt:\n{document[0].page_content[:200]}...\n")

# Load multiple documents from a directory
dir_loader = DirectoryLoader("sample_documents/", glob="*.txt", loader_cls=TextLoader)
documents = dir_loader.load()

print(f"Number of documents loaded: {len(documents)}")
for i, doc in enumerate(documents):
    print(f"Document {i+1} preview: {doc.page_content[:50]}...")

Content of sample1.txt:
# Comprehensive Overview of Artificial Intelligence

## Table of Contents
1. [Introduction to Artificial Intelligence](#introduction-to-artificial-intelligence)
2. [History of AI](#history-of-ai)
3. [...

Number of documents loaded: 1
Document 1 preview: # Comprehensive Overview of Artificial Intelligenc...


In [3]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader('sample_documents/sample2.pdf')
documents = loader.load()
print(f'PDF pages loaded: {len(documents)}')
print(f'First page preview:\n{documents[0].page_content[:200]}...')

PDF pages loaded: 26
First page preview:
Quiet-ST aR: Language Models Can T each Themselves to
Think Before Speaking
Eric Zelikman
Stanford University
Georges Harik
Notbad AI Inc
Yijia Shao
Stanford University
V aruna Jayasiri
Notbad AI Inc
...


## 2. Text Splitting and Chunking

In [4]:
# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

# Split the documents
splits = text_splitter.split_documents(documents)

print(f"Number of splits: {len(splits)}")
print(f"First split preview:\n{splits[0].page_content[:200]}...")

Number of splits: 112
First split preview:
Quiet-ST aR: Language Models Can T each Themselves to
Think Before Speaking
Eric Zelikman
Stanford University
Georges Harik
Notbad AI Inc
Yijia Shao
Stanford University
V aruna Jayasiri
Notbad AI Inc
...


In [5]:
# Create embeddings and FAISS vector store from the text splits
vectorstore = FAISS.from_documents(splits, embedding_model)
print(f'Vector store created with {vectorstore.index.ntotal} vectors')

Vector store created with 112 vectors


## 3. Building a Simple Question-Answering System

In [6]:
# Create LCEL retrieval chain (replaces deprecated RetrievalQA)
rag_prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}\nAnswer: "
)

rag_chain = (
    {"context": vectorstore.as_retriever(search_kwargs={"k": 3}), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Ask a question
query = "What is the main topic of these documents?"
answer = rag_chain.invoke(query)

print(f"Question: {query}")
print(f"Answer: {answer}\n")

Question: What is the main topic of these documents?
Answer: The main topic of these documents appears to be mathematics, specifically problem-solving and calculations involving quantities such as eggs, prices, and total earnings.



## 4. Implementing Semantic Search

In [7]:
# Perform a semantic search
query = "Discuss the importance of AI"
search_results = vectorstore.similarity_search(query, k=3)

print(f"Search query: {query}\n")
print("Top 3 relevant chunks:")
for i, doc in enumerate(search_results):
    print(f"Result {i+1}:\n{doc.page_content[:200]}...\n")

# Use the search results to answer a question
question = "What are some advantages of ai models?"
context = "\n".join([doc.page_content for doc in search_results])

prompt = f"Based on the following context, answer the question: {question}\n\nContext: {context}\n\nAnswer:"
answer = llm.invoke(prompt)

print(f"Question: {question}")
print(f"Answer: {answer}")

Search query: Discuss the importance of AI

Top 3 relevant chunks:
Result 1:
and GSM8K (Cobbe et al., 2021). On CommonsenseQA, we find that Quiet-STaR improves
performance by 10.9% compared to the base language model. As shown in Figure 2,
this improvement consistently increas...

Result 2:
token that affects attention and controls complex downstream behavior. In one related work,
Goyal et al. (2023) show that learning a single ”pause” token (essentially representing each
token as two to...

Result 3:
next step.
Now we will break down all the eggs from our starting number, $n$ =, into the pieces
that we set up previously. For the beginning number, we have: This brings us back
to our starting number...

Question: What are some advantages of ai models?
Answer: content="Based on the provided context, some advantages of AI models include:\n\n1. **Improved performance**: Quiet-STaR improves performance by 10.9% on CommonsenseQA and 5.0% on GSM8K compared to the base language model.\n\n2. **

## Conclusion

In this tutorial, we've explored various aspects of document processing with LangChain, including loading and parsing documents, text splitting, building a simple question-answering system, and implementing semantic search. These techniques form the foundation for more advanced document analysis and information retrieval systems.